# 


SPY Statistical Analysis, GARCH Volatility & 0DTE Option Chain

Pulls SPY daily bars via `schwab-py`, then:

1. **Descriptive stats** on the last 14 trading days
2. **GARCH(1,1)** conditional volatility fit on a longer lookback (14 days is too thin to fit a GARCH model reliably), forecasting the volatility currently "priced into" near-term price action
3. **0DTE option chain with Greeks** — falls back to the nearest upcoming expiration automatically if today isn't an actual SPY expiration day (Mon/Wed/Fri)

> **Note:** This notebook was authored and logic-tested against synthetic data in a sandbox with no network access to Schwab's API. You'll need to run it yourself with valid credentials — double check field names against your account's actual API response if Schwab has changed their schema since this was written.


## 0. Install dependencies

In [1]:
%pip install schwab-py arch statsmodels pandas numpy scipy matplotlib -q


Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [ ]:
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller



from arch import arch_model

import httpx
from schwab.auth import easy_client
from schwab.client import Client


## 2. Config

Fill in your app credentials. Both auth paths are handled by `easy_client`:
- If `TOKEN_PATH` already exists (you've authed before), it loads and auto-refreshes that token — no browser needed.
- If it doesn't exist, `easy_client` opens a browser for the OAuth login flow and writes a fresh token file to `TOKEN_PATH`.


In [6]:
api_key='RNz4Esx0WFVNTlGiP57GpJGO4XqLoAONtqC3dnrrG1MaXgJQ',
app_secret = '5SNiAxva0QjrwnGbcN0LEsQn33TBBj4OmjmPTutx3yw09UPtDDCebtBSAAnn3JzS',
callback_url='https://127.0.0.1:8182',
token_path='/tmp/token.json'     # created on first successful auth, reused after

SYMBOL = "SPY"

# Block 1: descriptive stats window
STATS_LOOKBACK_DAYS = 14

# Block 2: GARCH needs much more history than 14 obs to fit reliably.
# Pulling ~2 years of daily bars just for the GARCH fit; the 14-day stats
# block above stays on its own short window.
GARCH_LOOKBACK_YEARS = 2

# Block 3: option chain
STRIKE_COUNT = 20   # strikes above/below ATM to pull


## 3. Authenticate

In [9]:
c = easy_client(
    api_key='RNz4Esx0WFVNTlGiP57GpJGO4XqLoAONtqC3dnrrG1MaXgJQ',
    app_secret = '5SNiAxva0QjrwnGbcN0LEsQn33TBBj4OmjmPTutx3yw09UPtDDCebtBSAAnn3JzS',
    callback_url='https://127.0.0.1:8182',
    token_path='/tmp/token.json')

print("Authenticated.")



**************************************************************

This is the manual login and token creation flow for schwab-py.
Please follow these instructions exactly:

 1. Open the following link by copy-pasting it into the browser
    of your choice:

        https://api.schwabapi.com/v1/oauth/authorize?response_type=code&client_id=RNz4Esx0WFVNTlGiP57GpJGO4XqLoAONtqC3dnrrG1MaXgJQ&redirect_uri=https%3A%2F%2F127.0.0.1%3A8182&state=r3BrqMBfKp7ucnAN70gfg2ANYUZeUp

 2. Log in with your account credentials. You may be asked to
    perform two-factor authentication using text messaging or
    another method, as well as whether to trust the browser.

 3. When asked whether to allow your app access to your account,
    select "Allow".

 4. Your browser should be redirected to your callback URI. Copy
    the ENTIRE address, paste it into the following prompt, and press
    Enter/Return.

If you encounter any issues, see here for troubleshooting:
https://schwab-py.readthedocs.io/en/latest/au

Redirect URL>  https://127.0.0.1:8182/?code=C0.b2F1dGgyLmNkYy5zY2h3YWIuY29t.8eKPHjfCxzMFm8hRAEwIdQ3uaET6GoKG_7v0-11ZNcA%40&session=b63bd2ca-902f-4329-acca-782cf2b8765c&state=r3BrqMBfKp7ucnAN70gfg2ANYUZeUp


Authenticated.


## 4. Pull price history

Pulls `GARCH_LOOKBACK_YEARS` of daily bars in one call. The 14-day stats block just slices the tail of this same DataFrame, so we only make one price-history request.


In [10]:
period_map = {1: Client.PriceHistory.Period.ONE_YEAR, 2: Client.PriceHistory.Period.TWO_YEARS}
if GARCH_LOOKBACK_YEARS not in period_map:
    raise ValueError(f"GARCH_LOOKBACK_YEARS must be one of {list(period_map)}")

resp = c.get_price_history(
    SYMBOL,
    period_type=Client.PriceHistory.PeriodType.YEAR,
    period=period_map[GARCH_LOOKBACK_YEARS],
    frequency_type=Client.PriceHistory.FrequencyType.DAILY,
    frequency=Client.PriceHistory.Frequency.DAILY,
)
assert resp.status_code == httpx.codes.OK, f"price history request failed: {resp.status_code} {resp.text}"
history = resp.json()

candles = history["candles"]
price_df = pd.DataFrame(candles)
price_df["date"] = pd.to_datetime(price_df["datetime"], unit="ms")
price_df = price_df.set_index("date").sort_index()
price_df = price_df[["open", "high", "low", "close", "volume"]]

print(f"Pulled {len(price_df)} daily bars for {SYMBOL}, {price_df.index.min().date()} to {price_df.index.max().date()}")
price_df.tail()


Pulled 500 daily bars for SPY, 2024-09-03 to 2026-08-31


,open,high,low,close,volume
date,,,,,
2026-08-25 05:00:00,766.16,766.780,763.0501,765.91,27422263
2026-08-26 05:00:00,764.73,767.350,763.9300,766.08,28751592
2026-08-27 05:00:00,768.50,772.355,767.1600,771.10,34557064
2026-08-28 05:00:00,771.76,775.300,768.3076,769.35,36744344
2026-08-31 05:00:00,767.33,767.995,764.7150,767.05,38810779


---
## Block 1 — Descriptive statistics (last 14 trading days)


In [11]:
recent = price_df.tail(STATS_LOOKBACK_DAYS).copy()
log_ret_all = np.log(price_df["close"]).diff().dropna()
recent_log_ret = log_ret_all.tail(STATS_LOOKBACK_DAYS)

daily_mean = recent_log_ret.mean()
daily_std = recent_log_ret.std()
annualized_vol_realized = daily_std * np.sqrt(252)
skew = stats.skew(recent_log_ret)
kurt = stats.kurtosis(recent_log_ret)  # excess kurtosis
jb_stat, jb_p = stats.jarque_bera(recent_log_ret)
adf_stat, adf_p = adfuller(recent_log_ret)[0], adfuller(recent_log_ret)[1]

stats_summary = pd.DataFrame({
    "value": [
        recent["close"].iloc[-1],
        daily_mean,
        daily_std,
        annualized_vol_realized,
        skew,
        kurt,
        jb_stat,
        jb_p,
        adf_stat,
        adf_p,
    ]
}, index=[
    "last_close",
    "daily_mean_log_return",
    "daily_std_log_return",
    "annualized_realized_vol",
    "skew",
    "excess_kurtosis",
    "jarque_bera_stat",
    "jarque_bera_p",
    "adf_stat",
    "adf_p (returns stationarity)",
])
stats_summary


/var/folders/7h/wlc_mr3544s93pdz5_1nwmtm0000gn/T/ipykernel_29574/454977017.py:11: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, adf_p = adfuller(recent_log_ret)[0], adfuller(recent_log_ret)[1]
/var/folders/7h/wlc_mr3544s93pdz5_1nwmtm0000gn/T/ipykernel_29574/454977017.py:11: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_stat, adf_p = adfuller(recent_log_ret)[

,value
last_close,767.050000
daily_mean_log_return,-0.000326
daily_std_log_return,0.004750
annualized_realized_vol,0.075408
skew,-0.046358
excess_kurtosis,-1.014292
jarque_bera_stat,0.605141
jarque_bera_p,0.738916
adf_stat,0.023566
adf_p (returns stationarity),0.960438


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(recent.index, recent["close"], marker="o")
axes[0].set_title(f"{SYMBOL} close, last {STATS_LOOKBACK_DAYS} sessions")
axes[0].tick_params(axis="x", rotation=45)

axes[1].hist(recent_log_ret, bins=8, edgecolor="black")
axes[1].set_title("Log return distribution")
plt.tight_layout()
plt.show()


---
## Block 2 — GARCH(1,1) conditional volatility

Fit on the full `GARCH_LOOKBACK_YEARS` history (not just the 14-day window — GARCH needs a long series to estimate persistence and mean-reversion). Output is the model's forecast of volatility *currently priced into* the next session, which you can compare against the realized 14-day vol from Block 1.


In [ ]:
# arch_model wants returns scaled roughly to percent for numerical stability
scaled_ret = log_ret_all * 100

garch = arch_model(scaled_ret, vol="Garch", p=1, q=1, dist="t")
garch_res = garch.fit(disp="off")
print(garch_res.summary())


In [ ]:
forecast = garch_res.forecast(horizon=1, reindex=False)
next_day_var_pct2 = forecast.variance.values[-1, 0]     # variance in (percent-return)^2 units
next_day_vol_daily = np.sqrt(next_day_var_pct2) / 100     # back to decimal daily vol
next_day_vol_annualized = next_day_vol_daily * np.sqrt(252)

conditional_vol_annualized = (garch_res.conditional_volatility / 100) * np.sqrt(252)

print(f"GARCH(1,1) next-session forecast:")
print(f"  daily vol:       {next_day_vol_daily:.4%}")
print(f"  annualized vol:  {next_day_vol_annualized:.2%}")
print()
print(f"Realized annualized vol, last {STATS_LOOKBACK_DAYS} sessions: {annualized_vol_realized:.2%}")
print(f"GARCH long-run (unconditional) annualized vol: {(np.sqrt(garch_res.params['omega'] / (1 - garch_res.params['alpha[1]'] - garch_res.params['beta[1]'])) / 100) * np.sqrt(252):.2%}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(conditional_vol_annualized.index, conditional_vol_annualized.values, label="GARCH conditional vol (annualized)")
ax.axhline(annualized_vol_realized, color="orange", linestyle="--", label=f"realized vol, last {STATS_LOOKBACK_DAYS}d")
ax.axvspan(recent.index.min(), recent.index.max(), color="orange", alpha=0.1)
ax.set_title(f"{SYMBOL} GARCH(1,1) conditional volatility")
ax.legend()
plt.tight_layout()
plt.show()


---
## Block 3 — 0DTE option chain with Greeks

Looks for an expiration matching today's date. If today isn't a SPY expiration day, falls back to the nearest upcoming expiration and prints a warning so you know it's not actually 0DTE.


In [ ]:
resp = c.get_option_chain(
    SYMBOL,
    contract_type=Client.Options.ContractType.ALL,
    strike_count=STRIKE_COUNT,
    include_underlying_quote=True,
)
assert resp.status_code == httpx.codes.OK, f"option chain request failed: {resp.status_code} {resp.text}"
chain = resp.json()

underlying_price = chain.get("underlyingPrice")
print(f"{SYMBOL} underlying price: {underlying_price}")

today_str = datetime.date.today().isoformat()

def flatten_exp_map(exp_map, option_type):
    rows = []
    for exp_key, strikes in exp_map.items():
        exp_date = exp_key.split(":")[0]
        for strike_str, contracts in strikes.items():
            for contract in contracts:
                rows.append({
                    "expiration": exp_date,
                    "option_type": option_type,
                    "strike": float(strike_str),
                    "bid": contract.get("bid"),
                    "ask": contract.get("ask"),
                    "last": contract.get("last"),
                    "volume": contract.get("totalVolume"),
                    "open_interest": contract.get("openInterest"),
                    "iv": contract.get("volatility"),
                    "delta": contract.get("delta"),
                    "gamma": contract.get("gamma"),
                    "theta": contract.get("theta"),
                    "vega": contract.get("vega"),
                    "rho": contract.get("rho"),
                })
    return rows

rows = flatten_exp_map(chain.get("callExpDateMap", {}), "CALL") + flatten_exp_map(chain.get("putExpDateMap", {}), "PUT")
chain_df = pd.DataFrame(rows)

available_expirations = sorted(chain_df["expiration"].unique())

if today_str in available_expirations:
    target_exp = today_str
    print(f"0DTE found: expiration {target_exp}")
else:
    future_exps = [e for e in available_expirations if e >= today_str]
    if not future_exps:
        raise RuntimeError("No expirations returned on/after today — widen STRIKE_COUNT or check the response.")
    target_exp = future_exps[0]
    print(f"No 0DTE today ({today_str}). Falling back to nearest expiration: {target_exp}")

chain_today = chain_df[chain_df["expiration"] == target_exp].copy()
chain_today = chain_today.sort_values(["option_type", "strike"]).reset_index(drop=True)
chain_today


In [ ]:
# Near-the-money view: strikes within ~5% of the underlying price
if underlying_price:
    band = 0.05
    ntm = chain_today[chain_today["strike"].between(underlying_price * (1 - band), underlying_price * (1 + band))]
else:
    ntm = chain_today

cols = ["option_type", "strike", "bid", "ask", "last", "iv", "delta", "gamma", "theta", "vega", "rho", "volume", "open_interest"]
ntm[cols].style.format({
    "bid": "{:.2f}", "ask": "{:.2f}", "last": "{:.2f}",
    "iv": "{:.2%}", "delta": "{:.3f}", "gamma": "{:.4f}",
    "theta": "{:.3f}", "vega": "{:.3f}", "rho": "{:.3f}",
})
